# Notebook 02 — Dataset Splitting for skinX

**Input:** `skinX_unified_clean_light_skin_v1.csv` from Notebook 01. This dataset combines PAD-UFES Fitzpatrick I–II with MILK10k light-tone 4–5; the two skin-type scales are **not equivalent**.

**Goal:** Create and freeze reproducible training, validation and test sets, approximately **70% / 15% / 15%** by image count, while keeping each available group together and maintaining reasonable TARGET/NON_TARGET balance **within each source**.

- **PAD-UFES:** group by patient. **MILK10k:** group by lesion because patient IDs are not available. MILK10k cannot be claimed to have patient-independent splits.
- Splitting is performed once, before training. Every model uses the same saved files. No test results should be used for model or threshold selection.
- No images are copied, resized or modified here. Notebook 03 will read the CSVs and load the original images.


In [1]:
# 1. Connect Drive and load the frozen dataset from Notebook 01.
from google.colab import drive
from pathlib import Path
from IPython.display import display
import hashlib
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/SkinX")
MANIFEST_DIR = PROJECT_ROOT / "data" / "manifests"
SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
SEED = 42

manifest_path = MANIFEST_DIR / "skinX_unified_clean_light_skin_v1.csv"
if not manifest_path.is_file():
    raise FileNotFoundError(
        f"Frozen manifest not found: {manifest_path}\n"
        "Run Notebook 01's final validation and versioning cell first."
    )

df_unified = pd.read_csv(manifest_path)
required = ["image_id", "filepath", "dataset_source", "group_id", "grouping_level",
            "label_group", "binary_label", "md5_hash"]
missing = [col for col in required if col not in df_unified.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if df_unified[required].isna().any().any():
    raise ValueError("A required column contains missing values.")
if df_unified["image_id"].duplicated().any():
    raise ValueError("Duplicate image IDs found.")
if df_unified["md5_hash"].duplicated().any():
    raise ValueError("Exact duplicate images found (same MD5 hash).")
if not df_unified["binary_label"].isin([0, 1]).all():
    raise ValueError("Binary labels must be 0 or 1.")
if not (df_unified["binary_label"] == df_unified["label_group"].map({
    "NON_TARGET": 0, "TARGET": 1
})).all():
    raise ValueError("Binary labels do not match TARGET/NON_TARGET.")
if set(df_unified["dataset_source"]) != {"PAD_UFES_20", "MILK10K"}:
    raise ValueError("Unexpected or missing dataset source.")

print(f"Loaded frozen manifest: {len(df_unified)} images")
print("Source and class counts:")
display(pd.crosstab(df_unified["dataset_source"], df_unified["label_group"], margins=True))


Mounted at /content/drive
Loaded frozen manifest: 2333 images
Source and class counts:


label_group,NON_TARGET,TARGET,All
dataset_source,,,
MILK10K,332,997,1329
PAD_UFES_20,63,941,1004
All,395,1938,2333


In [2]:
# 2. Split each data source separately, keeping all images in a group together.
# 7 folds gives ~1/7 (14.3%) for test; 6 folds of what remains gives
# another ~14.3% for validation. Actual image counts may vary because groups differ in size.

def choose_holdout(frame, n_folds, seed):
    """Choose the most class-balanced fold; never split a group."""
    splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    expected = 1 / n_folds
    best_score = float("inf")
    best_indices = None

    for _, holdout_indices in splitter.split(
        np.zeros(len(frame)), frame["binary_label"], frame["group_id"]
    ):
        holdout = frame.iloc[holdout_indices]
        # Compare each class's held-out share against the desired share.
        score = abs(len(holdout) / len(frame) - expected)
        for label in [0, 1]:
            total = (frame["binary_label"] == label).sum()
            selected = (holdout["binary_label"] == label).sum()
            if total:
                score += abs(selected / total - expected)
                if selected == 0:
                    score += 10  # Avoid a split with no examples of a class.
        if score < best_score:
            best_score = score
            best_indices = holdout_indices

    return frame.iloc[best_indices].copy(), frame.drop(frame.index[best_indices]).copy()

parts = {"train": [], "val": [], "test": []}
for source, source_data in df_unified.groupby("dataset_source", sort=True):
    source_data = source_data.reset_index(drop=True)
    test, remaining = choose_holdout(source_data, n_folds=7, seed=SEED)
    val, train = choose_holdout(remaining.reset_index(drop=True), n_folds=6, seed=SEED)
    parts["train"].append(train)
    parts["val"].append(val)
    parts["test"].append(test)
    print(f"{source}: train={len(train)}, val={len(val)}, test={len(test)}")

splits = {}
for name in ["train", "val", "test"]:
    splits[name] = pd.concat(parts[name], ignore_index=True).sort_values(
        ["dataset_source", "image_id"]
    ).reset_index(drop=True)


MILK10K: train=950, val=189, test=190
PAD_UFES_20: train=711, val=150, test=143


In [3]:
# 3. Verify every image appears once, no group crosses splits, and both sources/classes exist.
all_ids = set(df_unified["image_id"])
assigned_ids = set()
all_groups = {}
all_hashes = {}

for name, frame in splits.items():
    ids = set(frame["image_id"])
    if len(ids) != len(frame) or ids & assigned_ids:
        raise ValueError(f"Repeated image ID inside or across splits: {name}")
    assigned_ids.update(ids)
    all_groups[name] = set(frame["group_id"])
    all_hashes[name] = set(frame["md5_hash"])
    if set(frame["binary_label"]) != {0, 1}:
        raise ValueError(f"Missing TARGET or NON_TARGET in {name}.")
    if set(frame["dataset_source"]) != {"PAD_UFES_20", "MILK10K"}:
        raise ValueError(f"Missing PAD or MILK in {name}.")
    if (pd.crosstab(frame["dataset_source"], frame["label_group"]) == 0).any().any():
        raise ValueError(f"A source/class combination is missing in {name}.")

if assigned_ids != all_ids:
    raise ValueError("Some input images were lost or were not assigned.")

for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    if all_groups[a] & all_groups[b]:
        raise ValueError(f"Group leakage between {a} and {b}.")
    if all_hashes[a] & all_hashes[b]:
        raise ValueError(f"Exact duplicate-image leakage between {a} and {b}.")

# Show overall split sizes and source/class proportions.
combined = pd.concat([
    frame.assign(split=name) for name, frame in splits.items()
], ignore_index=True)
summary = combined.groupby("split").agg(
    images=("image_id", "size"), groups=("group_id", "nunique"),
    target=("binary_label", "sum")
).reindex(["train", "val", "test"])
summary["non_target"] = summary["images"] - summary["target"]
summary["percentage"] = (100 * summary["images"] / len(df_unified)).round(1)

print("=== Split sizes (approximately 70 / 15 / 15) ===")
display(summary)
print("=== Source and class counts in each split ===")
display(pd.crosstab([combined["split"], combined["dataset_source"]],
                    combined["label_group"], margins=True))
print("No group or exact-duplicate leakage found.")
print("NOTE: MILK10k group IDs are lesion-level, NOT confirmed patient-level.")


=== Split sizes (approximately 70 / 15 / 15) ===


,images,groups,target,non_target,percentage
split,,,,,
train,1661,1331,1375,286,71.2
val,339,267,286,53,14.5
test,333,268,277,56,14.3


=== Source and class counts in each split ===


label_group           NON_TARGET  TARGET   All
split dataset_source                          
test  MILK10K                 47     143   190
      PAD_UFES_20              9     134   143
train MILK10K                240     710   950
      PAD_UFES_20             46     665   711
val   MILK10K                 45     144   189
      PAD_UFES_20              8     142   150
All                          395    1938  2333

No group or exact-duplicate leakage found.
NOTE: MILK10k group IDs are lesion-level, NOT confirmed patient-level.


In [4]:
# 4. Freeze the three split CSVs and their source fingerprint (never overwrite silently).
# If an existing v1 file differs, keep it unchanged and investigate or create v2.
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

metadata = {
    "input_manifest": manifest_path.name,
    "input_sha256": hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
    "seed": SEED,
    "method": "Source-wise StratifiedGroupKFold, 1 of 7 test, 1 of 6 validation",
    "grouping": {"PAD_UFES_20": "patient", "MILK10K": "lesion; patient IDs unavailable"},
    "images": {name: len(frame) for name, frame in splits.items()},
}

files_to_save = {
    SPLIT_DIR / f"skinX_{name}_manifest_v1.csv": frame.to_csv(index=False)
    for name, frame in splits.items()
}
files_to_save[SPLIT_DIR / "skinX_split_metadata_v1.json"] = (
    json.dumps(metadata, indent=2, sort_keys=True) + "\n"
)

# Check all existing files BEFORE writing any of them.
for path, content in files_to_save.items():
    if path.exists() and path.read_text(encoding="utf-8") != content:
        raise FileExistsError(
            f"Frozen file already exists with different contents: {path}\n"
            "Do not overwrite v1. Verify the source manifest or use a new version."
        )

for path, content in files_to_save.items():
    if not path.exists():
        path.write_text(content, encoding="utf-8")
    print(f"Saved / verified: {path.name}")

print("Notebook 02 complete. Use these exact frozen split files for all models.")


Saved / verified: skinX_train_manifest_v1.csv
Saved / verified: skinX_val_manifest_v1.csv
Saved / verified: skinX_test_manifest_v1.csv
Saved / verified: skinX_split_metadata_v1.json
Notebook 02 complete. Use these exact frozen split files for all models.
